# Building AI Teams: Multi-Agent Systems with Gemini

**Build with AI by GDG Gurugram | May 9, 2026**

Welcome! In the next 45 minutes, you will build a working **multi-agent AI system** using Gemini. Not a chatbot. A team of AI agents that collaborate to research a topic and write a report for you.

By the end of this notebook, you will have:

- a **Planner** agent that breaks down a topic into research questions
- a **Researcher** agent that searches the live web using Gemini's Google Search grounding
- a **Writer** agent that synthesizes everything into a final report
- an **orchestrator** that connects them into a working pipeline

You can keep using this for free after the workshop. Change the topic, extend the agents, ship it.

---

### How to use this notebook

1. Click **File → Save a copy in Drive** (top-left). Work on your copy, not the original.
2. Read each markdown cell, then run the code cell below it (Shift + Enter).
3. If something breaks, check the **Common errors** section in that step. We have seen most of them already.
4. The trainer will pause at checkpoints. If you finish a section early, try the **"Try this"** challenge at the end of it.

Let's go.


---

## Step 0: Setup (do this first, even before the workshop starts)

We need three things to start:
1. The Gemini Python SDK installed
2. A free API key from Google AI Studio
3. The key loaded into Colab

### 0.1 Install the SDK

Run the cell below. This installs the `google-genai` SDK (the new unified SDK — not the older `google-generativeai`).


In [ ]:
!pip install -q -U google-genai

### 0.2 Get a free Gemini API key

1. Open [aistudio.google.com/apikey](https://aistudio.google.com/apikey) in a new tab
2. Sign in with your Google account
3. Click **Create API key** → **Create API key in new project**
4. Copy the key. It looks like `AIza...`

The free tier is enough for this whole workshop. You will not be charged.

### 0.3 Add the key to Colab securely

In Colab, click the **🔑 key icon** in the left sidebar.

1. Click **Add new secret**
2. Name: `GEMINI_API_KEY` (exactly this, case-sensitive)
3. Value: paste your key
4. Toggle **Notebook access** ON

Then run the cell below. It loads your key into the environment.


In [ ]:
import os
from google.colab import userdata

# Load the API key from Colab secrets
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("API key loaded:", os.environ["GEMINI_API_KEY"][:8] + "..." + os.environ["GEMINI_API_KEY"][-4:])


### 0.4 Hello, Gemini

Quick smoke test. If this prints a sensible response, you are good to go.


In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hi to a workshop full of developers in one short sentence."
)

print(response.text)


**Expected output:** a one-sentence friendly hello.

#### Common errors

| Error | Fix |
|---|---|
| `SecretNotFoundError` | Your secret name is wrong. It must be exactly `GEMINI_API_KEY` and **Notebook access** must be ON. |
| `PermissionDenied: API key not valid` | The key you pasted is wrong or has a trailing space. Re-copy it from AI Studio. |
| `ResourceExhausted` / `429` | You hit the free-tier rate limit. Wait 30 seconds and re-run. |
| `ImportError: cannot import name 'genai'` | You are on the old SDK. Run the install cell again. The package is `google-genai`, not `google-generativeai`. |

✋ **Checkpoint.** Wait here until the trainer says move on.


---

## Step 1: What is an "agent", really?

Before we build, let's strip the hype.

> **An agent is just an LLM with a clearly defined role and a clear input/output contract.**

That's it. Every agent we build today is a Python function that:
1. Takes some input
2. Calls Gemini with a focused system instruction (its "role")
3. Returns a structured output

A **multi-agent system** is just multiple such functions, where the output of one becomes the input of the next.

We are NOT using LangChain, CrewAI, or any agent framework. We are writing the orchestration ourselves so you can see exactly how it works. Once you understand this, the frameworks become optional.

Let's build agent #1.


---

## Step 2: The Planner agent

**Role:** Take a user topic. Break it down into 3 sharp research questions.

**Why we need this:** if you ask an LLM "research X" in one shot, you get a vague essay. If you first decompose X into 3 specific questions and answer each one, you get a real report. This is how humans research too.

Run the cell below to define our Planner.


In [ ]:
import json

PLANNER_SYSTEM = """You are a research planner. Given a topic, you break it down into exactly 3 sharp, specific research questions that, when answered, would give someone a complete understanding of the topic.

Rules:
- Questions must be specific, not vague.
- Each question must be answerable with current web research.
- Cover different angles (e.g., what, why, how, who, when).

Return ONLY a JSON array of 3 strings. No prose, no markdown fences, just the JSON array.

Example for topic "should I learn Rust in 2026":
["What problems does Rust solve that other languages do not?", "Which companies and projects are hiring Rust developers in 2026?", "What is the realistic learning curve for a Python or JavaScript developer moving to Rust?"]
"""

def planner(topic: str) -> list[str]:
    """Take a topic, return 3 research questions."""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Topic: {topic}",
        config={"system_instruction": PLANNER_SYSTEM}
    )

    # The model returns a JSON array as text. Parse it.
    text = response.text.strip()
    # Strip markdown fences if the model wrapped them anyway
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    questions = json.loads(text.strip())
    return questions


### Try it

Run the cell below to see the Planner in action.


In [ ]:
topic = "is it worth using Claude Code for daily work in 2026"

questions = planner(topic)

print(f"Topic: {topic}\n")
print("Research questions:")
for i, q in enumerate(questions, 1):
    print(f"  {i}. {q}")


**Expected output:** 3 numbered research questions, each specific and different from the others.

#### What just happened

You wrote your first agent. Notice:

- The `system_instruction` is the agent's **role**. This is what makes it an "agent" instead of just an API call.
- The output contract is **JSON**. We chose this so the next agent can consume it programmatically.
- The function signature is `topic → list of questions`. Clean. Predictable. Testable.

#### Common errors

| Error | Fix |
|---|---|
| `JSONDecodeError` | The model wrapped output in ` ```json ` fences or added prose. Our code strips fences but if you changed the prompt, that handling may break. Print `response.text` to debug. |
| Empty list returned | Your topic was too vague. Try something more specific like "AI coding assistants in 2026" instead of just "AI". |

#### Try this (if you finish early)

Change the system prompt to ask for **5 questions** instead of 3. Re-run with a topic of your choice.

✋ **Checkpoint.**


---

## Step 3: The Researcher agent (with live Google Search)

This is the magic step. We are giving an agent a **tool**: the ability to search the live web through Gemini's built-in Google Search grounding.

**Role:** Take one research question. Search the web. Return an answer with sources.

This is where Gemini specifically shines — Google Search grounding is built right into the API. No scraping, no separate search service, no flaky third-party tools.


In [ ]:
from google.genai import types

RESEARCHER_SYSTEM = """You are a research analyst. You will be given one specific question. Search the web for current, factual information and write a focused 2-3 paragraph answer.

Rules:
- Be specific. Cite numbers, names, dates when relevant.
- Do not hedge with "it depends" without giving the actual factors.
- Do not invent facts. If you do not find good sources, say so.
"""

def researcher(question: str) -> dict:
    """Take a question, search the web, return an answer with sources."""

    # The Google Search tool — this is the key part.
    google_search_tool = types.Tool(google_search=types.GoogleSearch())

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=RESEARCHER_SYSTEM,
            tools=[google_search_tool],
        )
    )

    # Pull out the answer text
    answer = response.text

    # Pull out the sources Gemini used (if any)
    sources = []
    if response.candidates and response.candidates[0].grounding_metadata:
        chunks = response.candidates[0].grounding_metadata.grounding_chunks or []
        for chunk in chunks:
            if chunk.web:
                sources.append({"title": chunk.web.title, "uri": chunk.web.uri})

    return {"question": question, "answer": answer, "sources": sources}


### Try it

Let's run the Researcher on the first question our Planner produced earlier.


In [ ]:
result = researcher(questions[0])

print(f"Q: {result['question']}\n")
print(f"A: {result['answer']}\n")
print(f"Sources ({len(result['sources'])}):")
for s in result['sources'][:5]:
    print(f"  - {s['title']}")


**Expected output:** a 2-3 paragraph answer with current information, plus a list of source titles.

This is the moment people in the room usually go "wait, it actually searched the web?". Yes. That is what `google_search` as a tool does.

#### Common errors

| Error | Fix |
|---|---|
| `Tool 'google_search' not supported on this model` | You're on an older model. We use `gemini-2.5-flash` which supports it. |
| Empty `sources` list | Sometimes Gemini answers from its knowledge without searching. That's fine; the answer is still grounded in training data. |
| `429` rate limit | Free tier has per-minute limits. Wait 30 seconds. |

#### Why this matters

Most "AI agents" tutorials skip tools because they are messy to wire up. But an agent **without tools is just a clever prompt**. The moment you give an agent the ability to *do* something (search, read a file, call an API), you have a real agent.

✋ **Checkpoint.**


---

## Step 4: The Writer agent

**Role:** Take all the research and synthesize a clean, readable report.

**Why a separate agent?** Because the Researcher's job is to *find facts*. The Writer's job is to *structure prose*. These are different skills, and giving each agent one job makes each one better. This is the core lesson of multi-agent design.


In [ ]:
WRITER_SYSTEM = """You are a technical writer. You will receive a topic and a list of research findings (questions and answers). Synthesize them into a clear, well-structured report.

Format:
- Start with a 2-sentence executive summary.
- Use markdown headings for each major theme (do not just repeat the questions).
- End with a short "Bottom line" section with a clear recommendation or takeaway.
- Keep it under 400 words. Tight beats long.
- Do not invent facts. Use only what is in the research findings.
"""

def writer(topic: str, research: list[dict]) -> str:
    """Take topic + list of research results, return a final markdown report."""

    # Format the research into a clean prompt
    research_block = "\n\n".join([
        f"Q: {r['question']}\nA: {r['answer']}"
        for r in research
    ])

    prompt = f"Topic: {topic}\n\nResearch findings:\n\n{research_block}"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={"system_instruction": WRITER_SYSTEM}
    )

    return response.text


### Try it

We will run this in the next step as part of the full pipeline.

✋ **Checkpoint.**


---

## Step 5: Connect the team — the Orchestrator

This is the moment the multi-agent system comes together. The orchestrator is the function that runs the agents in the right order and passes data between them.

```
topic
  → planner → 3 questions
  → researcher (run on each question) → 3 answers
  → writer → final report
```

That's it. That is the entire multi-agent system.


In [ ]:
def research_agent_team(topic: str) -> str:
    """The full pipeline: planner → researcher (×3) → writer."""

    print(f"📋 Topic: {topic}\n")

    # Step 1: Planner breaks the topic into questions
    print("🧠 Planner is breaking down the topic...")
    questions = planner(topic)
    for i, q in enumerate(questions, 1):
        print(f"   {i}. {q}")
    print()

    # Step 2: Researcher answers each question
    print("🔎 Researcher is searching the web...")
    research = []
    for i, q in enumerate(questions, 1):
        print(f"   ({i}/{len(questions)}) {q[:60]}...")
        result = researcher(q)
        research.append(result)
    print()

    # Step 3: Writer synthesizes the final report
    print("✍️  Writer is drafting the report...\n")
    report = writer(topic, research)

    return report


### Run the full team

This is your moment. Pick a topic and let the agents do their thing. It will take 30-60 seconds because the Researcher runs 3 times.


In [ ]:
topic = "is it worth using Claude Code for daily work in 2026"

report = research_agent_team(topic)

print("=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(report)


You just ran a 3-agent system that takes a goal, plans, researches the live web, and writes a report. That is a multi-agent AI system.

Pause and let that land for a second.

#### Try this

Change the topic. Some good ones to try:

- `"what is the state of AI hardware competition in 2026"`
- `"should a small startup self-host an LLM or use APIs"`
- `"what are the most useful ChatGPT alternatives right now"`
- pick a topic from your own life

Run it again. The agents handle whatever you throw at them.

✋ **Checkpoint.**


---

## Step 6: Extensions — make it your own

You have the foundation. Here is how you'd extend it. (You don't need to do these in the workshop — they are for after.)

### Add a 4th agent: the Critic

A Critic agent reads the Writer's draft and either approves it or sends it back with feedback. This is how you get production-grade output.

```python
CRITIC_SYSTEM = """You are an editor. Read the report. Either approve it
(reply 'APPROVE') or list 2-3 specific improvements needed."""

def critic(report: str) -> str:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=report,
        config={"system_instruction": CRITIC_SYSTEM}
    )
    return response.text
```

Then loop: Writer drafts → Critic reviews → if not approved, Writer rewrites with feedback. This is called a **reflection loop** and it dramatically improves output quality.

### Add memory

Right now the agents have no memory across runs. To add memory, save each `research` list to a file (or a vector database like ChromaDB) and load relevant past research when a new topic comes in.

### Add more tools

The Researcher only has Google Search. You could add:
- **URL Context** (read a specific webpage): `types.Tool(url_context=types.UrlContext())`
- **Code Execution** (run python to compute things): `types.Tool(code_execution=types.ToolCodeExecution())`
- **Function calling** (call your own custom Python functions)

### Move to a real framework (when you're ready)

Once you understand the pattern, libraries like Google's [Agent Development Kit](https://google.github.io/adk-docs/) or LangGraph give you orchestration, state management, and tracing for free. But you wrote it yourself first, so you actually know what they are doing under the hood.

### Ship it

Wrap this notebook in a Streamlit app, deploy it on Hugging Face Spaces, you have a live AI research tool people can use. Total cost: $0.

---

## What you built today

- ✅ A 3-agent AI system
- ✅ Live web research via Gemini's Google Search tool
- ✅ A clean orchestration pattern you can extend forever
- ✅ Zero frameworks, zero magic, just Python + Gemini
- ✅ Free to run, free to share

Tag **@gdg_gurugram** and **@hitakshi** with what you build next 🚀

#BuildWithAI
